In [1]:
import cudf
import cuml
import cupy as cp
import matplotlib.pyplot as plt
import seaborn as sns
from cuml.model_selection import train_test_split
from sklearn.model_selection import KFold
from cuml.preprocessing import OneHotEncoder, LabelEncoder, RobustScaler
from cuml.decomposition import PCA
from cuml.cluster import DBSCAN
from cuml.neighbors import NearestNeighbors
from sklearn.metrics import (confusion_matrix, precision_score,
                           recall_score, f1_score, accuracy_score,
                           roc_auc_score)
from sklearn.model_selection import ParameterGrid
import numpy as np
import warnings
import kagglehub
import io

warnings.filterwarnings('ignore')

cuml.set_global_output_type('cupy')

In [2]:
def draw_confusion_matrix(cm, title='Confusion Matrix'):
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['Not Fraud', 'Fraud'],
                yticklabels=['Not Fraud', 'Fraud'])
    plt.title(title)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

def print_confusion_matrix(cm, title):
    print(f"{title}:")
    print("         Predicted")
    print("         Not Fraud  Fraud")
    print(f"Actual")
    print(f"Not Fraud  {cm[0,0]:6d}  {cm[0,1]:6d}")
    print(f"Fraud      {cm[1,0]:6d}  {cm[1,1]:6d}")

def plot_pca_clusters(pca_df, title):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='PC1', y='PC2', hue='fraud', style='fraud',
                    data=pca_df.to_pandas(), palette='viridis')
    plt.title(title)
    plt.xlabel('PC1')
    plt.ylabel('PC2')
    plt.show()

In [3]:
sample_data_str = """step,customer,age,gender,zipcodeOri,merchant,zipMerchant,category,amount,fraud
15,'C1520545421','1','M','28007','M151143676','28007','es_barsandrestaurants',43.12,0
42,'C320158645','2','M','28007','M348934600','28007','es_transportation',21.11,0
68,'C1886601622','3','M','28007','M348934600','28007','es_transportation',26.33,0
93,'C1872482068','4','M','28007','M349281107','28007','es_fashion',57.64,0
120,'C1175419672','2','M','28007','M1823072687','28007','es_transportation',36.10,0
7,'C1459406863','3','F','28007','M348934600','28007','es_transportation',18.75,0
55,'C2099817342','4','F','28007','M85975013','28007','es_health',89.50,0
112,'C1342850861','2','F','28007','M348934600','28007','es_transportation',32.40,0
29,'C1768391745','5','F','28007','M1823072687','28007','es_food',45.20,0
81,'C1029360548','3','F','28007','M348934600','28007','es_transportation',27.85,0
140,'C1987340562','1','M','28007','M1741626453','28007','es_travel',220.30,1
165,'C1430512409','4','F','28007','M980657600','28007','es_sportsandtoys',180.90,1
3,'C1093826151','2','M','28007','M348934600','28007','es_transportation',29.81,0
48,'C2051121374','3','F','28007','M1823072687','28007','es_transportation',15.45,0
76,'C173809379','5','M','28007','M855959430','28007','es_health',120.75,0
102,'C1293978246','1','F','28007','M348934600','28007','es_transportation',8.99,0
128,'C1530544995','3','M','28007','M1823072687','28007','es_transportation',35.67,0
154,'C2110252284','2','F','28007','M348934600','28007','es_barsandrestaurants',55.20,0
179,'C1041187332','4','M','28007','M348934600','28007','es_transportation',41.50,0
9,'C1782187339','3','F','28007','M1741626453','28007','es_leisure',65.80,0"""

In [4]:
def load_and_preprocess_data():
    path = kagglehub.dataset_download("ealaxi/banksim1")
    data = cudf.read_csv(path + '/bs140513_032310.csv')
    sample_data_df = cudf.read_csv(io.StringIO(sample_data_str))

    data = cudf.concat([data, sample_data_df], axis=0, ignore_index=True)
    data.columns = data.columns.str.strip("'").str.strip('"')

    data = data.drop(['zipcodeOri', 'zipMerchant'], axis=1)
    for col in ['customer', 'age', 'gender', 'merchant']:
        data[col] = data[col].str[1:-1]
    data['category'] = data['category'].str.replace('es_', '')
    data = data[data['gender'] != 'U']
    data['age'] = data['age'].replace('U', '7').astype('float32')
    data['amount'] = data['amount'].astype('float32')

    bins = [0, 500, 1000, 1500, 2000, 2500, 3000, float('inf')]
    labels = ['0-500', '500-1000', '1000-1500', '1500-2000',
              '2000-2500', '2500-3000', '>3000']
    data['amount_thresh'] = cudf.cut(data['amount'], bins=bins, labels=labels)

    for entity in ['customer', 'merchant', 'category', 'amount_thresh']:
        freq = data.groupby(entity).size().reset_index(name=f'{entity}_freq')
        freq[f'{entity}_ID'] = cp.arange(1, len(freq) + 1)
        data = data.merge(freq, on=entity)
        if entity in ['customer', 'merchant']:
            data = data.drop(entity, axis=1)
        else:
            data[entity] = data[f'{entity}_ID']
            data = data.drop(f'{entity}_ID', axis=1)

    total = len(data)
    for col in data.columns:
        if '_freq' in col:
            data[col] = (data[col] / total * 100).round(5)

    ohe = OneHotEncoder(sparse=True, handle_unknown='ignore')
    ohe_cols = ['age', 'gender', 'category', 'amount_thresh']
    ohe_data = ohe.fit_transform(data[ohe_cols])
    ohe_data = cudf.DataFrame(ohe_data.toarray(),
                             columns=ohe.get_feature_names(ohe_cols))
    data = cudf.concat([data.drop(ohe_cols, axis=1), ohe_data], axis=1)

    le = LabelEncoder()
    data['fraud'] = le.fit_transform(data['fraud'])

    return data

def evaluate_dbscan(params, X, y, n_splits=2):
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X).astype('float32')
    y = y.astype('int32')

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=123)
    fold_metrics = []

    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        train_idx = cp.array(train_idx)
        test_idx = cp.array(test_idx)

        X_train = X_scaled[train_idx]
        y_train = y.iloc[train_idx]
        X_test = X_scaled[test_idx]
        y_test = y.iloc[test_idx]

        dbscan = DBSCAN(eps=params['eps'],
                       min_samples=params['min_samples'],
                       output_type='cupy')

        train_labels = dbscan.fit_predict(X_train)

        core_mask = dbscan.core_sample_indices_
        if len(core_mask) == 0:
            test_labels = cp.full(X_test.shape[0], -1, dtype='int32')
            core_points = None
            core_labels = None
        else:
            core_points = X_train[core_mask]
            core_labels = train_labels[core_mask]

            nn = NearestNeighbors(n_neighbors=1)
            nn.fit(core_points)
            _, indices = nn.kneighbors(X_test)
            test_labels = core_labels[cp.array(indices.flatten())]

        def analyze_clusters(labels, true_labels):
            clusters = cudf.DataFrame({'cluster': labels, 'true_label': true_labels})
            stats = []
            unique_clusters = cp.unique(labels)
            for cluster in unique_clusters.get():
                if cluster == -1:
                    continue
                mask = clusters['cluster'] == cluster
                fraud_ratio = clusters[mask]['true_label'].mean().item()
                size = mask.sum().item()
                stats.append({
                    'cluster': cluster,
                    'fraud_ratio': fraud_ratio,
                    'size': size
                })
            return cudf.DataFrame(stats)

        train_stats = analyze_clusters(train_labels, y_train)
        cluster_mapping = {}
        for i in range(len(train_stats)):
            cluster = train_stats['cluster'].iloc[i].item()
            fraud_ratio = train_stats['fraud_ratio'].iloc[i].item()
            cluster_mapping[cluster] = 1 if fraud_ratio > params['fraud_threshold'] else 0
        cluster_mapping[-1] = 0

        def evaluate(y_true, labels, name, fold):
            pred = cudf.Series(labels).map(cluster_mapping).fillna(0)
            y_true_np = cp.asnumpy(y_true.values)
            y_pred_np = cp.asnumpy(pred.values)

            metrics = {
                'cm': confusion_matrix(y_true_np, y_pred_np),
                'recall': recall_score(y_true_np, y_pred_np, zero_division=0),
                'precision': precision_score(y_true_np, y_pred_np, zero_division=0),
                'f1': f1_score(y_true_np, y_pred_np, zero_division=0),
                'accuracy': accuracy_score(y_true_np, y_pred_np),
                'auc': roc_auc_score(y_true_np, y_pred_np)
            }

            print(f"\n=== {name} Fold {fold} Results ===")
            print_confusion_matrix(metrics['cm'], f"{name} Fold {fold} Confusion Matrix")
            print(f"Recall: {metrics['recall']:.4f} | Precision: {metrics['precision']:.4f}")
            print(f"F1: {metrics['f1']:.4f} | AUC: {metrics['auc']:.4f}")

            return metrics

        train_metrics = evaluate(y_train, train_labels, "Training", fold)
        test_metrics = evaluate(y_test, test_labels, "Test", fold)

        fold_metrics.append({
            'fold': fold,
            'train_metrics': train_metrics,
            'test_metrics': test_metrics,
            'cluster_stats': train_stats,
            'cluster_mapping': cluster_mapping,
            'core_points': core_points,
            'core_labels': core_labels,
            'dbscan': dbscan
        })
    avg_test_metrics = {
        'recall': np.mean([m['test_metrics']['recall'] for m in fold_metrics]),
        'precision': np.mean([m['test_metrics']['precision'] for m in fold_metrics]),
        'f1': np.mean([m['test_metrics']['f1'] for m in fold_metrics]),
        'accuracy': np.mean([m['test_metrics']['accuracy'] for m in fold_metrics]),
        'auc': np.mean([m['test_metrics']['auc'] for m in fold_metrics])
    }

    print(f"\n=== Average Test Metrics Across {n_splits} Folds ===")
    print(f"Recall: {avg_test_metrics['recall']:.4f} | Precision: {avg_test_metrics['precision']:.4f}")
    print(f"F1: {avg_test_metrics['f1']:.4f} | AUC: {avg_test_metrics['auc']:.4f}")

    return {
        'params': params,
        'fold_metrics': fold_metrics,
        'avg_test_metrics': avg_test_metrics,
        'scaler': scaler,
        'model': dbscan
    }

In [5]:
data = load_and_preprocess_data()
X = data.drop('fraud', axis=1)
y = data['fraud']

single_result = evaluate_dbscan(
    params={'eps': 0.75, 'fraud_threshold': 0.175, 'min_samples': 3},
    X=X,
    y=y,
    n_splits=2
)

last_fold = single_result['fold_metrics'][-1]
core_points = last_fold['core_points']
core_labels = last_fold['core_labels']
cluster_mapping = last_fold['cluster_mapping']

numeric_cols = X.select_dtypes(include=[cp.number]).columns
X_scaled_full = single_result['scaler'].transform(X[numeric_cols]).astype('float32')

if core_points is not None:
    nn = NearestNeighbors(n_neighbors=1)
    nn.fit(core_points)
    _, indices = nn.kneighbors(X_scaled_full)
    cluster_labels = core_labels[cp.array(indices.flatten())]
else:
    cluster_labels = cp.full(X_scaled_full.shape[0], -1, dtype='int32')

data['predicted_cluster'] = cudf.Series(cluster_labels)
data['predicted_fraud'] = data['predicted_cluster'].map(cluster_mapping).fillna(0).astype('int32')

[2025-09-02 20:04:08.845] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 8491 -> 7229. Using the larger integer type might result in better performance

=== Training Fold 0 Results ===
Training Fold 0 Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  293144     314
Fraud        3404     186
Recall: 0.0518 | Precision: 0.3720
F1: 0.0910 | AUC: 0.5254

=== Test Fold 0 Results ===
Test Fold 0 Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  291928    1508
Fraud        1177    2435
Recall: 0.6741 | Precision: 0.6176
F1: 0.6446 | AUC: 0.8345
[2025-09-02 20:04:40.762] [CUML] [info] Batch size limited by the chosen integer type (4 bytes). 8491 -> 7229. Using the larger integer type might result in better performance

=== Training Fold 1 Results ===
Training Fold 1 Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  293161     275
Fraud        3359     253
Recall: 0.0700 | Precis

In [6]:
start_idx = data.shape[0] - 20
print(data.iloc[start_idx::][['fraud', 'predicted_fraud']])

        fraud  predicted_fraud
594076      0                0
594077      0                0
594078      0                0
594079      0                0
594080      1                1
594081      1                0
594082      0                0
594083      0                0
594084      0                0
594085      0                0
594086      0                0
594087      0                0
594088      0                0
594089      0                0
594090      0                0
594091      0                0
594092      0                0
594093      0                0
594094      0                0
594095      0                0


In [7]:
def evaluate(y_true, predicted_fraud, name):
    y_true = cudf.Series(y_true)
    predicted_fraud = cudf.Series(predicted_fraud)

    y_true_np = cp.asnumpy(y_true.values)
    y_pred_np = cp.asnumpy(predicted_fraud.values)

    metrics = {
        'cm': confusion_matrix(y_true_np, y_pred_np),
        'recall': recall_score(y_true_np, y_pred_np, zero_division=0),
        'precision': precision_score(y_true_np, y_pred_np, zero_division=0),
        'f1': f1_score(y_true_np, y_pred_np, zero_division=0),
        'accuracy': accuracy_score(y_true_np, y_pred_np),
        'auc': roc_auc_score(y_true_np, y_pred_np)
    }

    print(f"\n=== {name} Results ===")
    print_confusion_matrix(metrics['cm'], f"{name} Confusion Matrix")
    print(f"Recall: {metrics['recall']:.4f} | Precision: {metrics['precision']:.4f}")
    print(f"F1: {metrics['f1']:.4f} | AUC: {metrics['auc']:.4f}")

    return metrics

In [8]:
y_true = data['fraud']
predicted_fraud = data['predicted_fraud']
metrics = evaluate(y_true, predicted_fraud, "Full Dataset")


=== Full Dataset Results ===
Full Dataset Confusion Matrix:
         Predicted
         Not Fraud  Fraud
Actual
Not Fraud  583681    3213
Fraud        2121    5081
Recall: 0.7055 | Precision: 0.6126
F1: 0.6558 | AUC: 0.8500
